<a href="https://colab.research.google.com/github/prachimishraa/GenAI/blob/main/Gen_AI_Lab_3_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from sklearn.datasets import make_blobs, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

class MultiClassNeuralNetwork:
    def __init__(self, layer_sizes, learning_rate=0.01):

        self.layer_sizes = layer_sizes
        self.lr = learning_rate
        self.weights = []
        self.biases = []

        for i in range(len(layer_sizes) - 1):
            w = np.random.randn(layer_sizes[i], layer_sizes[i+1]) * np.sqrt(2.0 / layer_sizes[i])
            b = np.zeros((1, layer_sizes[i+1]))
            self.weights.append(w)
            self.biases.append(b)

    def _relu(self, Z):
        return np.maximum(0, Z)

    def _relu_derivative(self, Z):
        return (Z > 0).astype(float)

    def _softmax(self, Z):
        exp_Z = np.exp(Z - np.max(Z, axis=1, keepdims=True))
        return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

    def forward(self, X):
        activations = [X]
        z_values = []

        A = X

        for i in range(len(self.weights) - 1):
            Z = np.dot(A, self.weights[i]) + self.biases[i]
            z_values.append(Z)
            A = self._relu(Z)
            activations.append(A)

        Z_out = np.dot(A, self.weights[-1]) + self.biases[-1]
        z_values.append(Z_out)
        A_out = self._softmax(Z_out)
        activations.append(A_out)

        return activations, z_values

    def compute_loss(self, Y_true, Y_pred):
        m = Y_true.shape[0]

        Y_pred = np.clip(Y_pred, 1e-15, 1 - 1e-15)
        return -np.sum(Y_true * np.log(Y_pred)) / m

    def backward(self, activations, z_values, Y_true):
        m = Y_true.shape[0]
        num_layers = len(self.layer_sizes) - 1

        dZ = activations[-1] - Y_true

        for l in reversed(range(num_layers)):
            A_prev = activations[l]

            dW = (1 / m) * np.dot(A_prev.T, dZ)
            db = (1 / m) * np.sum(dZ, axis=0, keepdims=True)

            if l > 0:
                dA_prev = np.dot(dZ, self.weights[l].T)
                dZ = dA_prev * self._relu_derivative(z_values[l-1])

            self.weights[l] -= self.lr * dW
            self.biases[l] -= self.lr * db

    def fit(self, X, Y_onehot, epochs=500, batch_size=32, verbose=True):
        num_samples = X.shape[0]
        for epoch in range(epochs):

            indices = np.arange(num_samples)
            np.random.shuffle(indices)
            X_shuffled = X[indices]
            Y_shuffled = Y_onehot[indices]

            for i in range(0, num_samples, batch_size):
                X_batch = X_shuffled[i:i+batch_size]
                Y_batch = Y_shuffled[i:i+batch_size]

                activations, z_values = self.forward(X_batch)
                self.backward(activations, z_values, Y_batch)

            if verbose and (epoch + 1) % (epochs // 5) == 0:
                full_activations, _ = self.forward(X)
                loss = self.compute_loss(Y_onehot, full_activations[-1])
                print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss:.4f}")

    def predict(self, X):
        activations, _ = self.forward(X)
        output_probs = activations[-1]
        return np.argmax(output_probs, axis=1)

X_raw, y_raw = make_classification(
    n_samples=1500, n_features=6, n_informative=4,
    n_classes=4, n_clusters_per_class=1, random_state=42
)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

encoder = OneHotEncoder(sparse_output=False)
Y_onehot = encoder.fit_transform(y_raw.reshape(-1, 1))

X_train, X_test, y_train, y_test, Y_train_oh, Y_test_oh = train_test_split(
    X_scaled, y_raw, Y_onehot, test_size=0.2, random_state=42
)

nn = MultiClassNeuralNetwork(layer_sizes=[6, 16, 8, 4], learning_rate=0.05)

print("Training Multi-Class Neural Network...")
nn.fit(X_train, Y_train_oh, epochs=300, batch_size=32)

test_preds = nn.predict(X_test)
accuracy = np.mean(test_preds == y_test) * 100
print(f"\nTest Set Accuracy: {accuracy:.2f}%")

Training Multi-Class Neural Network...
Epoch 60/300 - Loss: 0.3285
Epoch 120/300 - Loss: 0.2941
Epoch 180/300 - Loss: 0.3090
Epoch 240/300 - Loss: 0.2642
Epoch 300/300 - Loss: 0.2582

Test Set Accuracy: 84.00%
